In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
#%pip install protobuf

In [ ]:
#%pip install langchain

In [ ]:
#%pip install langchain-community

In [ ]:
#%pip install langchain-core

In [ ]:
#%pip install langchain-text-splitters

In [ ]:
#%pip install langchain-ollama

In [ ]:
#%pip install chromadb

In [ ]:
#%pip install unstructured

In [ ]:
#%pip install unstructured[pdf]

In [ ]:
#%pip install pypdf

In [ ]:
#%pip install ipython

In [ ]:
#%pip install "langchain==0.3.27"

In [ ]:
#%pip install -U transformers peft sentence-transformers --quiet

In [ ]:
#pip install  langchain-core

In [ ]:
#%pip uninstall -y langchain langchain-core langchain-community langchain-text-splitters pydantic pydantic-core


In [ ]:
#%pip install "pydantic>=2.6,<3" "langchain>=0.1.20,<0.2" 


In [ ]:
#pip install "langchain-core>=0.1.52,<0.2" "langchain-community>=0.0.28,<0.1"  "langchain-text-splitters>=0.0.1,<0.1"

In [2]:
# =========================
# Imports
# =========================
import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever

# Hugging Face / Transformers
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)
from langchain_community.llms import HuggingFacePipeline


In [3]:
# =========================
# Imports
# =========================
import os
import warnings
warnings.filterwarnings('ignore')

from IPython.display import display, Markdown

# Loaders
from langchain_community.document_loaders import (
    UnstructuredPDFLoader,
    UnstructuredWordDocumentLoader
)

# Splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Vector DB
from langchain_community.vectorstores import Chroma
from langchain_core.embeddings import Embeddings

# LangChain core
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain.retrievers.multi_query import MultiQueryRetriever

# Transformers / HF
import torch
from typing import List
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForCausalLM,
    pipeline
)
from langchain_community.llms import HuggingFacePipeline


In [4]:
# =========================
# Device
# =========================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

Device: cuda


In [5]:
# =========================
# Custom HF Embeddings (transformers-only)
# =========================
class HFTransformersEmbeddings(Embeddings):
    """
    Minimal embeddings using HuggingFace transformers only.
    Mean-pools last_hidden_state. Works on Kaggle (no sentence-transformers).
    """
    def __init__(self, model_name: str, device: str = "cpu", normalize: bool = True):
        self.device = device
        self.normalize = normalize
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def _embed(self, texts: List[str]) -> List[List[float]]:
        enc = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(self.device)

        outputs = self.model(**enc)
        last_hidden = outputs.last_hidden_state  # [B, T, H]
        attn_mask = enc["attention_mask"].unsqueeze(-1)  # [B, T, 1]
        masked = last_hidden * attn_mask
        sums = masked.sum(dim=1)
        counts = attn_mask.sum(dim=1).clamp(min=1)
        embs = sums / counts

        if self.normalize:
            embs = torch.nn.functional.normalize(embs, p=2, dim=1)

        return embs.cpu().tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        return self._embed(texts)

    def embed_query(self, text: str) -> List[float]:
        return self._embed([text])[0]

In [6]:
# =========================
# Configuration
# =========================
DATA_DIR = "/kaggle/input/datasets/princelubisi/dataset2-0"       # <--- change this to your folder
PERSIST_DIR = "./chroma_store_multi"        # persist your vector store
COLLECTION_NAME = "multi-doc-rag"

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"  # small, fast, works well on Kaggle
embedding_model = HFTransformersEmbeddings(
    model_name=EMBED_MODEL_NAME,
    device=DEVICE,
    normalize=True,
)
print(f"Custom embedding model loaded: {EMBED_MODEL_NAME}")

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/133M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Custom embedding model loaded: BAAI/bge-small-en-v1.5


In [7]:
# =========================
# Load ALL documents + metadata
# =========================
documents = []
supported_ext = (".pdf", ".docx")
file_list = [f for f in os.listdir(DATA_DIR) if f.lower().endswith(supported_ext)]
file_list.sort()

for filename in file_list:
    file_path = os.path.join(DATA_DIR, filename)

    if filename.lower().endswith(".pdf"):
        loader = UnstructuredPDFLoader(file_path=file_path)
    elif filename.lower().endswith(".docx"):
        loader = UnstructuredWordDocumentLoader(file_path=file_path)
    else:
        continue

    try:
        docs = loader.load()
        for d in docs:
            d.metadata["source"] = filename  # keep original filename as filter key
        documents.extend(docs)
        print(f"Loaded: {filename}")
    except Exception as e:
        print(f"Skipping {filename} due to loader error: {e}")

print(f"Total base documents loaded: {len(documents)}")

Loaded: FAQ - Declaration of Interest - final.pdf
Loaded: LLMs.pdf
Loaded: WEF_The_Global_Cooperation_Barometer_2024.pdf
Loaded: scammer-agent.pdf
Total base documents loaded: 4


In [8]:
# Optional: see available sources
ALL_SOURCES = sorted({d.metadata.get("source", "") for d in documents})
print("Discovered sources:", ALL_SOURCES)

Discovered sources: ['FAQ - Declaration of Interest - final.pdf', 'LLMs.pdf', 'WEF_The_Global_Cooperation_Barometer_2024.pdf', 'scammer-agent.pdf']


In [9]:
# =========================
# Split into chunks
# =========================
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 334


In [10]:
# =========================
# Build / Persist Vector DB
# =========================
# If you want to rebuild from scratch each run, you can clear the folder:
# import shutil; shutil.rmtree(PERSIST_DIR, ignore_errors=True)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)
vector_db.persist()
print(f"Vector DB created and persisted at: {PERSIST_DIR}")

Vector DB created and persisted at: ./chroma_store_multi


In [11]:
# =========================
# Load Mistral‑7B‑Instruct v0.3 (quantized if possible)
# =========================
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Try 4-bit quantization for VRAM savings
try:
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Loaded Mistral in 4‑bit.")
except Exception as e:
    print(f"4‑bit not available ({e}). Falling back to fp16/fp32.")
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=dtype,
        device_map="auto",
        max_memory={0: "13GiB", "cpu": "32GiB"} if DEVICE == "cuda" else None,
    )
    print(f"Loaded Mistral in {dtype} with offloading if needed.")

# Reduce VRAM during long prompts
model.generation_config.use_cache = False

gen_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.05,
    return_full_text=False,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

llm = HuggingFacePipeline(pipeline=gen_pipe)
print("Mistral‑7B‑Instruct pipeline ready.")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'pad_token_id', 'repetition_penalty', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Loaded Mistral in torch.float16 with offloading if needed.
Mistral‑7B‑Instruct pipeline ready.


In [12]:
# =========================
# Multi-Query Retriever
# =========================
MULTI_QUERY_PROMPT = PromptTemplate(
    input_variables=["question"],
    template=(
        "You are an AI language model assistant. Your task is to generate 3 "
        "different versions of the given user question to retrieve relevant documents from "
        "a vector database. Provide these alternative questions separated by newlines.\n\n"
        "Original question: {question}"
    ),
)

# Base retriever across *all* documents
base_retriever = vector_db.as_retriever(search_kwargs={"k": 3})

multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm,
    prompt=MULTI_QUERY_PROMPT
)

In [13]:
# =========================
# RAG Prompt
# =========================
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a helpful assistant. Answer the question based ONLY on the following context.
If the answer is not contained in the context, say: "I don't know based on the provided document."

Context:
{context}

Question: {question}

Answer:
"""
)

In [14]:
# =========================
# Token budgeted context
# =========================
def fit_context_by_tokens(texts, tokenizer, budget_tokens=3000):
    kept, total = [], 0
    for t in texts:
        n = len(tokenizer.encode(t))
        if total + n > budget_tokens:
            break
        kept.append(t)
        total += n
    return "\n\n".join(kept)


def build_context(question, doc_name=None, budget_tokens=3000, top_k=3):
    """
    If doc_name is provided, restrict retrieval to that file via metadata filter.
    Otherwise, use multi-query retriever across all docs.
    """
    if doc_name:
        # Filter to the specific source
        retriever = vector_db.as_retriever(
            search_kwargs={"k": top_k, "filter": {"source": doc_name}}
        )
        docs = retriever.get_relevant_documents(question)
    else:
        docs = multi_retriever.get_relevant_documents(question)

    texts = [d.page_content for d in docs]
    return fit_context_by_tokens(texts, tokenizer, budget_tokens=budget_tokens)

In [16]:
# =========================
# Final chain
# =========================
def chat_with_documents(question, doc_name=None, budget_tokens=3000):
    """
    Chat over multiple documents (optionally restricted to `doc_name`).
    """
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    context = build_context(question, doc_name=doc_name, budget_tokens=budget_tokens)
    prompt = RAG_PROMPT.format(context=context, question=question)

    output = llm(prompt)
    return display(Markdown(output))

In [17]:
# =========================
# Examples
# =========================
chat_with_documents("what is a conflict of interest ?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



A conflict of interest describes the situation in which personal (private) or financial/business interests of an individual may unduly influence their decisions which they are to make independently and objectively on behalf of another individual or legal entity.

In [18]:
chat_with_documents("Who must complete a Declaration of Interest?")

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Salaried Staff, Supervisors/Specialists & Managers of VWGA and its subsidiaries must complete a Declaration of Interest if a potential conflict exists.